# 01 — Exploratory Data Analysis (MovieLens 25M)

This notebook explores the MovieLens 25M dataset to understand:
- Rating distribution
- User activity patterns
- Item popularity (long-tail distribution)
- Sparsity of the interaction matrix
- Temporal patterns in ratings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Load Data

In [ ]:
ratings = pd.read_csv('../data/ml-25m/ratings.csv')
movies = pd.read_csv('../data/ml-25m/movies.csv')

print(f'Ratings: {len(ratings):,} rows')
print(f'Movies:  {len(movies):,} rows')
print(f'Users:   {ratings["userId"].nunique():,}')
print(f'Items:   {ratings["movieId"].nunique():,}')
print(f'\nSparsity: {1 - len(ratings) / (ratings["userId"].nunique() * ratings["movieId"].nunique()):.4%}')

ratings.head()

## 2. Rating Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating value distribution
ratings['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Rating Values')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Rating over time
ratings['date'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings.set_index('date').resample('M')['rating'].count().plot(ax=axes[1], color='steelblue')
axes[1].set_title('Ratings Over Time (Monthly)')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Number of Ratings')

plt.tight_layout()
plt.savefig('../figures/rating_distribution.png', dpi=150)
plt.show()

print(f'Mean rating: {ratings["rating"].mean():.2f}')
print(f'Median rating: {ratings["rating"].median():.2f}')

## 3. User Activity Distribution

In [ ]:
user_counts = ratings.groupby('userId').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
user_counts.clip(upper=500).plot(kind='hist', bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('User Activity Distribution (clipped at 500)')
axes[0].set_xlabel('Number of Ratings per User')

# Log scale
user_counts.plot(kind='hist', bins=100, ax=axes[1], color='steelblue', edgecolor='white', log=True)
axes[1].set_title('User Activity Distribution (log scale)')
axes[1].set_xlabel('Number of Ratings per User')

plt.tight_layout()
plt.savefig('../figures/user_activity.png', dpi=150)
plt.show()

print(f'Mean ratings/user:   {user_counts.mean():.1f}')
print(f'Median ratings/user: {user_counts.median():.1f}')
print(f'Min:                 {user_counts.min()}')
print(f'Max:                 {user_counts.max()}')

## 4. Item Popularity — Long Tail Distribution

In [ ]:
item_counts = ratings.groupby('movieId').size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(len(item_counts)), item_counts.values, color='steelblue', linewidth=0.5)
ax.fill_between(range(len(item_counts)), item_counts.values, alpha=0.3, color='steelblue')
ax.set_xlabel('Movie Rank (by popularity)')
ax.set_ylabel('Number of Ratings')
ax.set_title('Item Popularity — Long Tail Distribution')
ax.set_yscale('log')

# Mark the 80/20 point
cumsum = item_counts.cumsum() / item_counts.sum()
idx_80 = (cumsum >= 0.8).idxmax()
rank_80 = list(item_counts.index).index(idx_80)
ax.axvline(x=rank_80, color='red', linestyle='--', label=f'80% of ratings from top {rank_80} movies')
ax.legend()

plt.tight_layout()
plt.savefig('../figures/long_tail.png', dpi=150)
plt.show()

print(f'Top 1% of movies account for {item_counts.head(int(len(item_counts)*0.01)).sum()/item_counts.sum():.1%} of all ratings')

## 5. Genre Analysis

In [ ]:
# Explode genres
movies_genres = movies.copy()
movies_genres['genres'] = movies_genres['genres'].str.split('|')
movies_genres = movies_genres.explode('genres')

# Merge with ratings
genre_ratings = ratings.merge(movies_genres[['movieId', 'genres']], on='movieId')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Genre count
genre_counts = genre_ratings['genres'].value_counts()
genre_counts.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Number of Ratings by Genre')
axes[0].set_xlabel('Count')

# Average rating by genre
genre_avg = genre_ratings.groupby('genres')['rating'].mean().sort_values(ascending=True)
genre_avg.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Average Rating by Genre')
axes[1].set_xlabel('Mean Rating')
axes[1].axvline(x=ratings['rating'].mean(), color='black', linestyle='--', label='Overall Mean')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/genre_analysis.png', dpi=150)
plt.show()

## 6. Key Insights

1. **Severe sparsity (~99.7%)** — most user-item pairs have no rating, making collaborative filtering challenging
2. **Long-tail distribution** — a small fraction of movies receive the majority of ratings (cold-start problem for unpopular items)
3. **Rating skew** — ratings skew positive (mean ~3.5), so binarization threshold of 3.5 gives balanced positive/negative classes
4. **Active users** — user activity varies widely (20 to 10,000+ ratings), requiring strategies like negative sampling
5. **Genre patterns** — Drama and Comedy dominate; Film-Noir and Documentary have highest average ratings